In [1]:
!pip install langchain_community

In [2]:
!pip install langchain_huggingface

In [3]:
!pip install langchain_chroma


In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [5]:
!pip install sentence-transformers chromadb

In [8]:
import pandas as pd

books = pd.read_csv("/content/books_cleaned.csv")

In [9]:
books["tagged_description"].to_csv(
    "tagged_description.txt",
    sep="\n",
    index=False,
    header=False
)

In [10]:
raw_documents = TextLoader("tagged_description.txt").load()


In [16]:
text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=1000,
    chunk_overlap=0
)

In [17]:
documents = text_splitter.split_documents(raw_documents)
documents[0]

Document(metadata={'source': 'tagged_description.txt'}, page_content='9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gi

In [22]:
persist_dir = "chroma_books_db"
import os
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

if not os.path.exists(persist_dir):
    print("⏳ Creating embeddings... (this happens only once)")

    db_books = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_dir
    )

    print("Database created and saved!")

else:
    print("Loading existing database (fast)")

    db_books = Chroma(
        persist_directory=persist_dir,
        embedding_function=embedding_model
    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading existing database (fast)


In [25]:
len(documents)

2932

In [26]:
db_books._collection.count()

0

In [27]:
import shutil
import os

if os.path.exists("chroma_books_db"):
    shutil.rmtree("chroma_books_db")

In [31]:
import shutil
import os

persist_dir = "chroma_books_db"
new_persist_dir = "chroma_books_db_new" # Use a new directory name to avoid lingering locks

# Ensure a clean slate before creating a new database
# Remove both old and potentially new directories for a clean run
if os.path.exists(persist_dir):
    shutil.rmtree(persist_dir)
    print(f"Removed existing {persist_dir} directory.")
if os.path.exists(new_persist_dir):
    shutil.rmtree(new_persist_dir)
    print(f"Removed existing {new_persist_dir} directory.")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db_books = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory=new_persist_dir # Use the new directory
)
print(f"Chroma database created at {new_persist_dir}.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chroma database created at chroma_books_db_new.


In [32]:
query = "A book to teach children about nature"
docs = db_books.similarity_search(query, k=10)
docs

[Document(id='e4699c67-cb75-47b6-8986-fa5e5f776c90', metadata={'source': 'tagged_description.txt'}, page_content="9780786808069 Children will discover the exciting world of their own backyard in this introduction to familiar animals from cats and dogs to bugs and frogs. The combination of photographs, illustrations, and fun facts make this an accessible and delightful learning experience.\n9780786808373 Introducing your baby to birds, cats, dogs, and babies through fine art, illsutration and photographs. These books are a rare opportunity to expose little ones to a range of images on a single subject, from simple child's drawings and abstract art to playful photos. A brief text accompanies each image, introducing baby to some basic -- and sometimes playful -- information on the subjects."),
 Document(id='5ee75838-854f-4288-8e58-b0a4eb6cd99f', metadata={'source': 'tagged_description.txt'}, page_content="9780064402453 ‘Racso, a brash and boastful little rodent, is making his way to Thorn

In [33]:
books[books["isbn13"] == int(docs[0].page_content.split()[0].strip())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
3747,9780786808069,0786808063,Baby Einstein: Neighborhood Animals,Marilyn Singer;Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=X9a4P...,Children will discover the exciting world of t...,2001.0,3.89,16.0,180.0,Baby Einstein: Neighborhood Animals,9780786808069 Children will discover the excit...


In [38]:
def retrieve_semantic_recommendations(
    query: str,
    top_k: int = 5,
) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k=top_k)

    books_list = []

    for rec in recs:
        isbn = int(rec.page_content.strip('"').split()[0])
        books_list.append(isbn)

    return books[books["isbn13"].isin(books_list)]

In [39]:
retrieve_semantic_recommendations("A book to teach children about nature")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
404,9780064402453,0064402452,Racso and the Rats of NIMH,Jane Leslie Conly,Juvenile Fiction,http://books.google.com/books/content?id=MgoNv...,"‘Racso, a brash and boastful little rodent, is...",1988.0,3.76,288.0,3231.0,Racso and the Rats of NIMH,"9780064402453 ‘Racso, a brash and boastful lit..."
816,9780142402931,0142402931,The Far Side of Evil,Sylvia Engdahl,Juvenile Fiction,http://books.google.com/books/content?id=7nijj...,A young girl from an advanced civilization is ...,2005.0,3.98,324.0,57.0,The Far Side of Evil,9780142402931 A young girl from an advanced ci...
3747,9780786808069,0786808063,Baby Einstein: Neighborhood Animals,Marilyn Singer;Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=X9a4P...,Children will discover the exciting world of t...,2001.0,3.89,16.0,180.0,Baby Einstein: Neighborhood Animals,9780786808069 Children will discover the excit...
3749,9780786808380,0786808381,Baby Einstein: Babies,Julie Aigner-Clark,Juvenile Fiction,http://books.google.com/books/content?id=jv4NA...,"Introduce your babies to birds, cats, dogs, an...",2002.0,4.03,20.0,29.0,Baby Einstein: Babies,"9780786808380 Introduce your babies to birds, ..."
4898,9781593851170,1593851170,The Nature of Play,Anthony D. Pellegrini;Peter K. Smith,Psychology,http://books.google.com/books/content?id=Nukz6...,"""Comprehensive and up to date, this tightly ed...",2005.0,4.25,308.0,4.0,The Nature of Play: Great Apes and Humans,"9781593851170 ""Comprehensive and up to date, t..."


I think we need improvement in recommendations. I will switch to zero shot classification to enhance the process.
